# User and Entity Behaviour Analytics (UEBA): Logon Anomaly Detection

## Background

Insider-threat investigations may involve looking for unusual patterns in employees’ system activity. User and Entity Behaviour Analytics (UEBA) uses behavioural data to help identify activity that differs from established patterns. Such differences can have legitimate explanations, so an anomaly should be treated as a signal for further review rather than proof of wrongdoing.

This project uses the CERT Insider Threat Dataset to explore user-level logon behaviour. Although the notebook also loads device and file-activity data, the anomaly-detection model uses logon records only. The other data sources are not combined with the model in this version of the project.

## Goal and Motivation

The aim is to build a simple, unsupervised workflow for identifying user profiles with unusual logon patterns. The analysis creates behavioural features, applies Isolation Forest to flag outliers, reviews flagged users in the context of their roles, and uses Principal Component Analysis (PCA) to visualise the feature data.

Isolation Forest does not require labelled examples of malicious behaviour. Instead, it identifies profiles that differ from the wider group according to the selected features. Therefore, a flagged profile should be interpreted as unusual within this dataset, not as evidence that the user is malicious.

## 1. Exploratory Data Analysis

In [1]:
import pandas as pd
import numpy as np

logon = pd.read_csv('logon.csv')
device = pd.read_csv('device.csv')
file = pd.read_csv('file.csv')
users = pd.read_csv('users.csv')

In [2]:
print(logon.head(10))
print(logon.info())
print(logon.describe())

                         id                 date     user       pc activity
0  {F3X8-Y2GT43DR-4906OHBL}  01/02/2010 02:19:18  DNS1758  PC-0414    Logon
1  {B4Q0-D0GM24KN-3704MAII}  01/02/2010 02:31:12  DNS1758  PC-0414   Logoff
2  {T7J1-D4HK34KV-5476TCIJ}  01/02/2010 02:34:02  DNS1758  PC-5313    Logon
3  {S4Y6-D8MQ05SA-0759HLIS}  01/02/2010 02:53:30  DNS1758  PC-5313   Logoff
4  {F3P0-E7FH78CV-4874FRGZ}  01/02/2010 04:07:31  DNS1758  PC-0012    Logon
5  {M6C2-B1LK96JR-3409QEXP}  01/02/2010 04:10:34  DNS1758  PC-0012   Logoff
6  {J3S1-G0CN29NY-6126GYJV}  01/02/2010 06:16:00  ANC1950  PC-4921    Logon
7  {M1V4-S1FK13BQ-2952XBLU}  01/02/2010 06:25:00  SAB1954  PC-5091    Logon
8  {S2E1-Y8HO73WG-6712ZKOG}  01/02/2010 06:28:00  LIM1718  PC-7539    Logon
9  {X3U5-I6AY76TL-5507LCII}  01/02/2010 06:32:00  LSN1672  PC-9021    Logon
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3530285 entries, 0 to 3530284
Data columns (total 5 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   id 

In [3]:
print(device.head(10))
print(device.info())
print(device.describe())

                         id                 date     user       pc  \
0  {Z2Q8-K3AV28BE-9353JIRT}  01/02/2010 07:17:18  SDH2394  PC-5849   
1  {C7F1-G7LE60RU-2483DAXS}  01/02/2010 07:22:42  JKS2444  PC-6961   
2  {T9A4-D4RV69OF-1704NINW}  01/02/2010 07:31:42  CBA1023  PC-1570   
3  {S8L0-O6QQ15NL-0636OYNV}  01/02/2010 07:33:28  GNT0221  PC-6427   
4  {U0F1-R1FX27FM-6954TTVU}  01/02/2010 07:33:55  JKS2444  PC-6961   
5  {X4R9-W7HH90OA-2624LIUE}  01/02/2010 07:37:13  SDH2394  PC-5849   
6  {F7Z4-U0BJ54IA-1102TDYV}  01/02/2010 07:40:11  RCT1697  PC-5770   
7  {N0N4-S0KF96GO-9894WIGG}  01/02/2010 07:41:02  ROR3483  PC-4365   
8  {F0B4-J9BQ17DC-9497NDHP}  01/02/2010 07:41:31  RCT1697  PC-5770   
9  {M1J9-R5HR77HW-9205YFCI}  01/02/2010 07:42:43  DJA0740  PC-7197   

                                           file_tree    activity  
0                          R:\;R:\22B5gX4;R:\SDH2394     Connect  
1                                     R:\;R:\JKS2444     Connect  
2  R:\;R:\42gY283;R:\48rr4y2

In [4]:
print(file.head(10))
print(file.info())
print(file.describe())

                         id                 date     user       pc  \
0  {F3E2-X3MV05YQ-3516SZDT}  01/02/2010 07:19:41  SDH2394  PC-5849   
1  {I6N1-Z7VL92UY-8715ESKQ}  01/02/2010 07:21:30  SDH2394  PC-5849   
2  {G4X5-J7MH70FV-8936QVSB}  01/02/2010 07:22:11  SDH2394  PC-5849   
3  {M2M7-Z5ST21EU-6704NSKO}  01/02/2010 07:24:06  SDH2394  PC-5849   
4  {R0A9-O9XB25PE-9236MALV}  01/02/2010 07:24:45  SDH2394  PC-5849   
5  {T1C9-A7BW31ZW-0408TSMQ}  01/02/2010 07:25:10  SDH2394  PC-5849   
6  {Y0N7-E6EJ23LX-3486GHCH}  01/02/2010 07:26:30  SDH2394  PC-5849   
7  {D5J4-S3AZ71KE-8915PYTL}  01/02/2010 07:27:03  SDH2394  PC-5849   
8  {U4B5-M9PO56AC-1319NFVD}  01/02/2010 07:27:13  SDH2394  PC-5849   
9  {Z4Z8-G2VU05EV-8184SUQX}  01/02/2010 07:27:59  SDH2394  PC-5849   

                  filename     activity  to_removable_media  \
0          R:\60WBQE7S.doc    File Open               False   
1          R:\0VGILDW8.pdf   File Write                True   
2          R:\60WBQE7S.doc    File Copy 

In [5]:
print(users.head(10))
print(users.info())
print(users.describe())

              employee_name  user_id                              email  \
0  Nicholas Fletcher Pruitt  NFP2441  Nicholas.Fletcher.Pruitt@dtaa.com   
1     Abraham Dante Rodgers  ADR1517     Abraham.Dante.Rodgers@dtaa.com   
2     Medge Wilma Blackburn  MWB4000     Medge.Wilma.Blackburn@dtaa.com   
3     Meghan Laurel Salazar  MLS2856     Meghan.Laurel.Salazar@dtaa.com   
4          Beau Todd Romero  BTR2026          Beau.Todd.Romero@dtaa.com   
5      Zephania Chaim Terry  ZCT1009      Zephania.Chaim.Terry@dtaa.com   
6       Sierra Emily Mendez  SEM1983       Sierra.Emily.Mendez@dtaa.com   
7          Bell Daria White  BDW1459          Bell.Daria.White@dtaa.com   
8     Fulton August Jackson  FAJ3101     Fulton.August.Jackson@dtaa.com   
9           Carl Louis Rice  CLR1239           Carl.Louis.Rice@dtaa.com   

                   role    projects  business_unit  \
0               ITAdmin         NaN              1   
1  ProductionLineWorker         NaN              1   
2  Productio

In [6]:
print(logon.isnull().sum())

id          0
date        0
user        0
pc          0
activity    0
dtype: int64


In [7]:
print(device.isnull().sum())

id                0
date              0
user              0
pc                0
file_tree    773474
activity          0
dtype: int64


In [8]:
print(file.isnull().sum())

id                      0
date                    0
user                    0
pc                      0
filename                0
activity                0
to_removable_media      0
from_removable_media    0
content                 0
dtype: int64


In [9]:
print(users.isnull().sum())

employee_name         0
user_id               0
email                 0
role                  0
projects           3317
business_unit         0
functional_unit       2
department           22
team                158
supervisor            1
start_date            0
end_date              0
dtype: int64


## 2. Construct Time Features in the Logon Dataset

The following features describe when each recorded event occurred. In this notebook, an event is classified as after-hours if it occurs before 06:00 or from 21:00 onward. This is an analysis choice for the project, rather than a universal definition of working hours.

In [10]:
logon['date'] = pd.to_datetime(logon['date'], format='%m/%d/%Y %H:%M:%S')
logon['hour'] = logon['date'].dt.hour
logon['day_of_week'] = logon['date'].dt.dayofweek
logon['is_weekend'] = (logon['day_of_week'] >= 5).astype(int)
# After-hours is defined here as before 06:00 or from 21:00 onward.
logon['after_hours'] = ((logon['hour'] < 6) | (logon['hour'] > 20)).astype(int)

## 3. Build User-Level Logon Activity Profiles

The event records are aggregated by user to create one profile per user. The profile contains the total number of logon-dataset events, the proportion of events occurring on weekends, the proportion occurring after hours, and the number of distinct PCs associated with the user. The event count includes both `Logon` and `Logoff` records.

In [11]:
logon_profile = logon.groupby('user').agg(
    logon_activity_events=('activity', 'count'),
    weekend_ratio=('is_weekend', 'mean'),
    after_hours_ratio=('after_hours', 'mean'),
    unique_pc_count=('pc', 'nunique')
)

In [ ]:
logon_profile.describe()

## 4. Detect Unusual User Profiles with Isolation Forest

Isolation Forest is an unsupervised algorithm that identifies observations that are easier to isolate from the rest of the data. Here, it is applied to the scaled user-level profiles to flag users whose selected logon features differ from the wider population.

## A) Scaling Data

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

logon_profile_scaled = scaler.fit_transform(logon_profile)

## B) Fit the Isolation Forest Model

The model uses 200 trees and a contamination setting of 0.01. This setting tells the algorithm to expect approximately 1% of profiles to be outliers; it is a modelling parameter, not an estimate of the actual prevalence of insider threats.

In [ ]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42
)

# contamination=0.01 sets the model's expected outlier fraction to approximately 1%.
iso.fit(logon_profile_scaled)

## C) Predict Anomalies and add results back to table

In [ ]:
logon_profile['anomaly_label'] = iso.predict(logon_profile_scaled)

## D) Convert to Anomaly Scores and add results back to table

In [ ]:
logon_profile['anomaly_score'] = iso.decision_function(logon_profile_scaled)

## E) Inspect Users with the Lowest Anomaly Scores

The users with the lowest anomaly scores are the profiles the model considers most unusual. These results can help prioritise further inspection, but they do not establish whether the activity is malicious.

In [ ]:
logon_profile.sort_values('anomaly_score').head(10)

In [ ]:
logon_profile.sort_values('unique_pc_count', ascending=False).head(20)

## 5. Merge the Logon Profiles with User Information

The user-level anomaly results are joined with the user metadata so that flagged profiles can be reviewed by role, department, and other available organisational attributes. This adds context to the results, but role information alone cannot determine whether an event is legitimate or suspicious.

In [ ]:
logon_users = (
    logon_profile.reset_index().merge(
        users,
        left_on='user',
        right_on='user_id',
        how='left'
    )
)

logon_users = logon_users.fillna('Unknown')

In [ ]:
logon_users[
    logon_users['anomaly_label']==-1
][[
    'user',
    'role',
    'department',
    'team',
    'unique_pc_count'
]]

## 6. Initial Interpretation of the Anomaly Detection Results

The flagged profiles differ from the wider group across the features used by the model. In the saved results, flagged users tend to have high event counts, higher proportions of after-hours activity, and a large number of distinct PCs. Many of the flagged users are IT Administrators, including users in Electronic Security teams. Their roles may help explain these patterns, so the results should be investigated in context rather than interpreted as confirmed threats.

## 7. Examine Anomaly Rates by Role

Comparing anomaly rates across roles helps show whether the model flags some job groups more often than others. Differences may reflect the responsibilities or access requirements of each role, as well as potentially unusual behaviour. These rates are descriptive and do not establish which group is more likely to pose an insider threat.

In [ ]:
role_summary = (
    logon_users.groupby('role')['anomaly_label'].apply(
        lambda x: (x == -1).mean()
    ).sort_values(ascending=False)
)

print(role_summary.head(10))

In [ ]:
final_summary = (
    logon_users.groupby('role')['anomaly_label'].value_counts(normalize=True)
)

print(final_summary)

## 8. Principal Component Analysis (PCA)

PCA reduces the feature data to a smaller number of dimensions so that broad patterns can be visualised. It is used here to explore the four selected logon features, not as a separate measure of anomaly-detection performance.

## A) Build the Feature Matrix

The PCA analysis uses the same four user-level features as the Isolation Forest model: event count, weekend activity ratio, after-hours activity ratio, and unique PC count.

In [ ]:
features = ['logon_activity_events', 'weekend_ratio', 'after_hours_ratio', 'unique_pc_count']

x = logon_profile[features]

## B) Scale Features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_scaled = scaler.fit_transform(x)

## C) PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

x_pca = pca.fit_transform(x_scaled)

print(x_pca.shape)

## D) Create Dataframe

In [ ]:
pca_df = pd.DataFrame(x_pca, columns=['PC1', 'PC2'])

pca_df['anomaly_label'] = logon_profile['anomaly_label'].values

pca_df['user'] = logon_profile.index

## E) Plotting and Visualizing

In [ ]:
import matplotlib.pyplot as plt

normal = pca_df[pca_df['anomaly_label'] == 1]
anomaly = pca_df[pca_df['anomaly_label'] == -1]

plt.figure(figsize=(8,6))

plt.scatter(
    normal['PC1'],
    normal['PC2'],
    alpha=0.4,
    s=20,
    label='Normal'
)

plt.scatter(
    anomaly['PC1'],
    anomaly['PC2'],
    color='red',
    s=80,
    edgecolor='black',
    label='Anomaly'
)

for _, row in anomaly.iterrows():
    plt.text(row['PC1'], row['PC2'], row['user'], fontsize=8)

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.title('PCA Projection of User Behavior')

plt.legend()

plt.show()

## F) Inspect Explained Variance Ratio

In [ ]:
print(pca.explained_variance_ratio_)
print(pca.explained_variance_ratio_.sum())

## G) Inspect PCA's Loadings

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=features
)

print(loadings)

## H) Interpret the PCA Results

The first two principal components explain approximately **93.75%** of the variance in the four standardised features. The first component is associated with event count, after-hours activity, and unique PC count, while the second is dominated by weekend activity ratio.

This means the two-dimensional projection retains much of the variation in these features. It does not show that the model is accurate or that flagged users are malicious.

## 9. Conclusion

This project demonstrates a basic UEBA workflow using logon-derived features, Isolation Forest, role-based review, and PCA. The model identifies user profiles that differ from the wider group, and the saved results show that many flagged profiles belong to IT Administrator roles, including Electronic Security teams. These patterns may reflect legitimate work responsibilities, which highlights the importance of contextual investigation.

The first two PCA components explain approximately 93.75% of the variance in the selected features, providing a compact way to visualise user profiles. However, the notebook does not validate anomaly flags against known insider-threat cases, and it does not test a real-time detection process. The results should therefore be treated as exploratory findings rather than evidence of confirmed threats or production-ready detection.

## 10. Limitations

This notebook has several limitations. First, the anomaly-detection model uses logon-derived features only; device and file-activity data are not integrated into the model. Second, the user profiles aggregate records across the available observation period, so the analysis does not simulate a detector that uses only information available at a particular point in time. Third, the model’s `contamination=0.01` setting influences how many profiles are flagged and should not be interpreted as the true prevalence of insider threats.

The model is unsupervised, and the notebook does not validate its flags against known insider-threat cases. Legitimate job responsibilities may also explain unusual patterns, particularly for IT and security staff. Finally, PCA explained variance describes how much feature variation is retained in the projection; it is not a measure of threat-detection accuracy. The device and user datasets also contain missing values that would need to be handled if these sources were incorporated into future modelling.

## 11. Future Work

Future work could combine logon data with device, file-access, and email activity to create richer behavioural profiles. It could also use time-windowed features to examine changes in behaviour over time, compare Isolation Forest with other anomaly-detection methods, and evaluate anomaly rankings against known insider-threat scenarios where suitable labels are available. Role-aware baselines could also be explored to provide context for different job responsibilities, while avoiding the assumption that everyone in the same role behaves identically.